In [ ]:
!pip install openai-agents

In [ ]:
from agents import Agent, Runner, AsyncOpenAI, ModelSettings, OpenAIChatCompletionsModel, set_tracing_disabled

In [ ]:
# Disable tracing/logging for the agents 
set_tracing_disabled(disabled=True)

In [ ]:
# Initialize an asynchronous OpenAI client, pointing to Groq's API endpoint
client = AsyncOpenAI(base_url="https://api.groq.com/openai/v1", api_key="gsk_YOUR-API_KEY")

In [ ]:
# Create an AI Agent for generating recipes
recipe_agent = Agent(
    name="Recipe Agent",   # Name for the agent
    instructions=(         # Instructions that define the agent's role
        "You are an agent for creating recipes. "
        "You will be given the name of a food and your job "
        "is to output that as an actual detailed recipe. "
        "The cooking time should be in minutes."
    ),
    model=OpenAIChatCompletionsModel(  # Set the model to use for generating completions
        model="llama-3.3-70b-versatile",  # Groq-hosted LLaMA model
        openai_client=client              # Pass the previously created async client
    )
)

# Define the recipe topic (input for the agent)
topic = "Italian pasta"

# Run the agent asynchronously using Runner
result = await Runner.run(recipe_agent, topic)

# Print the final generated recipe
print(result.final_output)

In [ ]:
language_agent = Agent(
    name="Language Agent",
    instructions="You are a language expert. You are given a recipe and you need to rewrite it in a different language.",
    model = OpenAIChatCompletionsModel(
        model="llama-3.3-70b-versatile",
        openai_client=client    
    ),
)

result = await Runner.run(recipe_agent, topic)
translated_result = await Runner.run(language_agent, f"Translate this recipe to Spanish: {result.final_output}")
print(f"Original Recipe:\n{result.final_output}\n")
print(f"Translated Recipe:\n{translated_result.final_output}")

In [ ]:
from pydantic import BaseModel
from pprint import pprint

# Import Pydantic BaseModel for defining a structured data model
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    instructions: list[str]
    cooking_time: int
    servings: int


# Create an agent for generating structured recipes
recipe_agent = Agent(
    name="Recipe Agent",   
    instructions=(         
        "You are an agent for creating recipes. You will be given the name of a food and your job"
        " is to output that as an actual detailed recipe. The cooking time should be in minutes."
    ),
    output_type=Recipe,    # Specify the Pydantic model for structured output
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  
        openai_client=client  
    ),
)

topic = "Italian pasta"

response = await Runner.run(recipe_agent, topic, max_turns=30)
recipe = response.final_output
recipe # Display the recipe object (will show in structured format because it's a Pydantic model)

In [ ]:
from agents import Agent, FileSearchTool, Runner, WebSearchTool, OpenAIResponsesModel

INSTRUCTIONS = (
    "You are a research assistant. Given a search term, you search the web for that term and produce a concise summary of the results."
    "The summary must 2-3 paragraphs and less than 300 words."
)

agent = Agent(
    name="search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model=OpenAIResponsesModel(
        model="openai/gpt-oss-20b",  
        openai_client=client  
    ),
    model_settings=ModelSettings(tool_choice="required")
)

async def main():
    result = await Runner.run(agent, "Virat kohli")
    print(result.final_output)

In [ ]:
from agents import Agent, function_tool

# Define a tool function that the agent can call
@function_tool
def get_weather(city: str) -> str:
    """
    Simulates retrieving the current weather for a given city.

    This function prints a message indicating that it is fetching weather
    information and returns a fixed example weather report. It is intended
    for demonstration purposes only and does not query real weather data.

    Args:
        city (str): The name of the city to get the weather for.

    Returns:
        str: A simulated weather report for the specified city.
    """
    
    print(f"Getting weather for {city}")
    
    return f"The weather in {city} is sunny, 28°C."


# Create a weather assistant agent
agent = Agent(
    name="Weather Agent",  
    instructions="You are a helpful weather assistant.",  
    tools=[get_weather],  # List of tool functions available to the agent
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3-32b",  
        openai_client=client  
    )
)

result = await Runner.run(agent, "Mumbai")
print(result.final_output)

In [ ]:
# Create an agent that translates text to Spanish
spanish_agent = Agent(
    name="Spanish agent",  
    instructions="You translate the user's message to Spanish",  
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3-32b",  
        openai_client=client  
    )
)

# Create an agent that translates text to French
french_agent = Agent(
    name="French agent",  
    instructions="You translate the user's message to French",  
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3-32b",  
        openai_client=client  
    )
)

# Create an orchestrator agent that can call the Spanish or French translation agents as tools
orchestrator_agent = Agent(
    name="orchestrator_agent",  
    instructions=(
        "You are a translation agent. You use the tools given to you to translate. "
        "If asked for multiple translations, you call the relevant tools."
    ),
    tools=[
        # Register the Spanish agent as a tool
        spanish_agent.as_tool(
            tool_name="translate_to_spanish",  
            tool_description="Translate the user's message to Spanish",  
        ),
        # Register the French agent as a tool
        french_agent.as_tool(
            tool_name="translate_to_french",  
            tool_description="Translate the user's message to French",  
        ),
    ],
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3-32b",  
        openai_client=client  
    )
)

result = await Runner.run(orchestrator_agent, input="Say 'Hello, how are you?' in Spanish.")
print(result.final_output)

In [ ]:
result = await Runner.run(orchestrator_agent, input="Say 'Hello, how are you?' in French.")
print(result.final_output)

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "gsk_YOUR-API_KEY"

In [ ]:
import os
print("GROQ key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("Prefix:", os.getenv("GROQ_API_KEY")[:8] if os.getenv("GROQ_API_KEY") else None)

In [ ]:
import os
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    handoff,
    RunContextWrapper,
    OpenAIChatCompletionsModel,
)

os.environ["GROQ_API_KEY"] = "gsk_YOUR-API_KEY"

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

history_tutor_agent = Agent(
    name="History Tutor",
    handoff_description="Specialist agent for historical questions",
    instructions="You provide assistance with historical queries. Explain important events and context clearly.",
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        openai_client=client
    ),
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You provide assistance with math queries. Explain your reasoning at each step and include examples.",
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        openai_client=client
    ),
)

def on_math_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to math tutor agent")

def on_history_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to history tutor agent")

history_handoff = handoff(history_tutor_agent, on_handoff=on_history_handoff)
math_handoff = handoff(math_tutor_agent, on_handoff=on_math_handoff)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You determine which agent to use based on the user's homework question. "
        "Use the history tutor for history questions. "
        "Use the math tutor for math questions. "
        "If neither agent is relevant, provide a general response."
    ),
   tools=[
    history_tutor_agent.as_tool(tool_name="history_tutor", tool_description="Use for history questions"),
    math_tutor_agent.as_tool(tool_name="math_tutor", tool_description="Use for math questions"),
],
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        openai_client=client
    ),
)

result = await Runner.run(triage_agent, "who was maharana pratap?")
print(result.final_output)

In [ ]:
result = await Runner.run(triage_agent, "what is 28 + 12 ?")
print(result.final_output)

In [ ]:
from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)

# Define the structured output for the guardrail check
class MathHomeworkOutput(BaseModel):
    is_math_homework: bool  # Flag to indicate if the input is math homework
    reasoning: str          # Explanation of why it was classified as math homework

# Agent responsible for detecting if the input is a math homework question
guardrail_agent = Agent(
    name="Guardrail check",  
    instructions="Check if the user is asking you to do their math homework.",  
    output_type=MathHomeworkOutput,  
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        openai_client=client 
    )
)


# Input guardrail function to intercept inputs before they reach the main agent
@input_guardrail
async def math_guardrail(
    ctx: RunContextWrapper[None],  # Context for the current run
    agent: Agent,                  # Reference to the calling agent
    input: str | list[TResponseInputItem]  # User input (string or structured)
) -> GuardrailFunctionOutput:
    
    # Run the guardrail check agent to classify the input
    result = await Runner.run(guardrail_agent, input, context=ctx.context)

    # Return guardrail decision
    return GuardrailFunctionOutput(
        output_info=result.final_output,  # Pass along structured output for reference
        tripwire_triggered=result.final_output.is_math_homework,  # Tripwire if homework detected
    )

# Main customer support agent with math homework guardrail enabled
agent = Agent(
    name="Customer support agent",  
    instructions="You are a customer support agent. You help customers with their questions.",
    input_guardrails=[math_guardrail],  
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  
        openai_client=client  
    )
)

# Main execution function to test the guardrail
async def main():
    try:
        # Attempt to process a math homework query
        await Runner.run(agent, "Hello, can you help me solve for x: 2x + 3 = 11?")
        print("Guardrail didn't trip - this is unexpected")  # This should not happen for math homework

    except InputGuardrailTripwireTriggered:
        # Expected behavior when math homework is detected
        print("Math homework guardrail tripped")

In [ ]:
await main()

In [ ]:
async def main():
    try:
        result = await Runner.run(agent, "who is Sachin Tendulkar")
        print(result.final_output)
        print("\n Guardrail didn't trip")

    except InputGuardrailTripwireTriggered:
        print("Math homework guardrail tripped")

await main()

In [ ]:
import os
from pydantic import BaseModel
from openai import AsyncOpenAI
from agents import (
    Agent,
    GuardrailFunctionOutput,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    output_guardrail,
    OpenAIChatCompletionsModel,
)

os.environ["GROQ_API_KEY"] = "gsk_YOUR-API_KEY"

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

# Define the structured format for the main agent's output
class MessageOutput(BaseModel): 
    response: str  # The text response from the agent

# Define the structured format for the guardrail check output
class MathOutput(BaseModel): 
    reasoning: str  # Explanation of why it was classified as math
    is_math: bool   # Flag indicating whether the content contains math

# Agent responsible for detecting if the output contains math
guardrail_agent = Agent(
    name="Guardrail check",  
    instructions="Check if the output includes any math.",  
    output_type=MathOutput,  
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  
        openai_client=client  
    )
)

# Output guardrail function that runs after the main agent produces a response
@output_guardrail
async def math_guardrail(
    ctx: RunContextWrapper,       # Context for the current run
    agent: Agent,                 # Reference to the calling agent
    output: MessageOutput         # The structured output from the main agent
) -> GuardrailFunctionOutput:

    # Check the agent's output to see if it contains math
    result = await Runner.run(guardrail_agent, output.response, context=ctx.context)

    # Return guardrail decision
    return GuardrailFunctionOutput(
        output_info=result.final_output,        # Pass along structured output for reference
        tripwire_triggered=result.final_output.is_math,  # Tripwire if math detected
    )

# Main customer support agent with math output guardrail enabled
agent = Agent( 
    name="Customer support agent",  
    instructions="You are a customer support agent. You help customers with their questions.",
    output_guardrails=[math_guardrail],  
    output_type=MessageOutput,           
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  
        openai_client=client  
    )
)

# Main function to test the output guardrail
async def main():
    # This input should trigger the guardrail
    try:
        await Runner.run(agent, "Hello, can you help me solve for x: 2x + 3 = 11?")
        print("Guardrail didn't trip - this is unexpected")  # Should not happen if math detected

    except OutputGuardrailTripwireTriggered:
        # Expected behavior when math is detected in output
        print("Math output guardrail tripped")

In [ ]:
await main()

In [ ]:
async def main():
    try:
        result = await Runner.run(agent, "who is virat kohli")
        print(result.final_output)
        print("\n Guardrail didn't trip")

    except OutputGuardrailTripwireTriggered:
        print("Math output guardrail tripped")

await main()

In [ ]:
import os
from dataclasses import dataclass
from pydantic import BaseModel
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    TResponseInputItem,
    OpenAIChatCompletionsModel,
)

os.environ["GROQ_API_KEY"] = "gsk_YOUR-API_KEY"

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

@dataclass
class UserProfile:
    id: str
    name: str

agent = Agent[UserProfile](
    name="Assistant",
    instructions="Reply very concisely.",
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  
        openai_client=client  
    )
)

profile = UserProfile(id="123", name="Nitesh")
print("You are now chatting with assistant. Type 'exit' to end the conversation.")
convo_items: list[TResponseInputItem] = []


while True:
    user_input = input("You: ")

    if user_input == "exit":
        print("Goodbye!")
        break

    convo_items.append({"content": user_input, "role": "user"})
    result = await Runner.run(agent, convo_items, context=profile)
    
    print(f"Assistant: {result.final_output}")
    
    convo_items = result.to_input_list()

In [ ]:
import os
from agents import Agent, Runner, SQLiteSession,  OpenAIChatCompletionsModel
from openai import AsyncOpenAI

os.environ["GROQ_API_KEY"] = "gsk_YOUR-API_KEY"

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

# Create agent
agent = Agent(
    name="Assistant",
    instructions="Reply very concisely.",
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  
        openai_client=client  
    )
)

# Create a session instance with a session ID
session = SQLiteSession("conversation_123")

# First turn
result = await Runner.run(
    agent,
    "What city is the Golden Gate Bridge in?",
    session=session
)
print(result.final_output)

# Second turn - agent automatically remembers previous context
result = await Runner.run(
    agent,
    "What state is it in?",
    session=session
)
print(result.final_output)  # "California"

In [ ]:
from agents import SQLiteSession

# In-memory database (lost when process ends)
session = SQLiteSession("user_123")

# Persistent file-based database
session = SQLiteSession("user_123", "conversations.db")

# Use the session
result = await Runner.run(
    agent,
    "Hello",
    session=session
)

In [ ]:
from agents import Agent, Runner, SQLiteSession

agent = Agent(
    name="Assistant", 
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  
        openai_client=client  
    ))

# Different sessions maintain separate conversation histories
session_1 = SQLiteSession("user_123", "conversations.db")
session_2 = SQLiteSession("user_456", "conversations.db")

result1 = await Runner.run(
    agent,
    "Hello my name is Prem",
    session=session_1
)
result2 = await Runner.run(
    agent,
    "Hello my name is Prakash",
    session=session_2
)

print(result1.final_output)
print(result2.final_output)

In [ ]:
result1 = await Runner.run(
    agent,
    "Hi what is my name",
    session=session_1
)
result2 = await Runner.run(
    agent,
    "Hi what is my name",
    session=session_2
)

print(result1.final_output)
print(result2.final_output)

In [ ]:
from openai.types.responses import ResponseTextDeltaEvent

agent = Agent(
    name="Joker",
    instructions="You are a helpful assistant.",
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        openai_client=client
    )
)

In [ ]:
# Run the agent in streaming mode so responses are generated and returned in chunks
result = Runner.run_streamed(agent, "Please tell me 5 jokes")

# Iterate asynchronously over the streamed events from the agent
async for event in result.stream_events():
    # Check if the event is a raw text delta event (partial text output)
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        # Print the partial text without adding a newline, flush to display immediately
        print(event.data.delta, end="", flush=True)
        

In [ ]:
!pip install "openai-agents[viz]"

In [ ]:
from agents import Agent, function_tool, handoff, RunContextWrapper
from agents.extensions.visualization import draw_graph
from pydantic import BaseModel


@function_tool
def print_something():
    print("blah blah")

history_tutor_agent = Agent(
    name="History Tutor",
    handoff_description="Specialist agent for historical questions",
    instructions="You provide assistance with historical queries. Explain important events and context clearly.",
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You provide assistance with math queries. Explain your reasoning at each step and include examples"
)

def on_math_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to math tutor agent")

def on_history_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to history tutor agent")

# This agent has the capability to handoff to either the history or math tutor agent
triage_agent = Agent(
    name="Triage Agent",
    instructions="You determine which agent to use based on the user's homework question." +
    "If neither agent is relevant, provide a general response.",
    handoffs=[handoff(history_tutor_agent, on_handoff=on_history_handoff), 
              handoff(math_tutor_agent, on_handoff=on_math_handoff)],
    tools=[print_something]
)

In [7]:
draw_graph(triage_agent)

NameError: name 'triage_agent' is not defined

In [1]:
!pip install "openai-agents[viz]"

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

In [6]:
import os, shutil
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"
print(shutil.which("dot"))

None
